# Music Genre Classification using Long Short-Term Memory (LSTM)
This notebook implements an **MFCC-based Long Short-Term Memory (LSTM)** neural network for music genre classification on the GTZAN dataset.

The preprocessing pipeline, 3-second audio segmentation, 40-MFCC feature extraction, song-level data splitting (70% train / 15% validation / 15% test), and feature scaling are kept strictly consistent with the other project architectures (MLP, 1D CNN, and Transformer) to ensure a fair comparative benchmark.


## 1. Imports


In [1]:
import os
import re
import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

print('TensorFlow Version:', tf.__version__)
print('Librosa Version:', librosa.__version__)


TensorFlow Version: 2.21.0
Librosa Version: 0.11.0


## 2. Configuration
Define global hyperparameters, audio processing parameters, and random seeds for reproducible execution.


In [2]:
# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)
tf.keras.utils.set_random_seed(RANDOM_SEED)

# Audio Preprocessing Parameters
SAMPLE_RATE = 22050
DURATION_PER_SEGMENT = 3  # seconds
N_MFCC = 40
N_FFT = 2048
HOP_LENGTH = 512

# Training Parameters
BATCH_SIZE = 32
MAX_EPOCHS = 30
EARLY_STOPPING_PATIENCE = 5
LEARNING_RATE = 0.001

# Results output directory
RESULTS_DIR = 'results'
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Configuration configured successfully.')


Configuration configured successfully.


## 3. Dataset Loading
Load the GTZAN 3-second metadata CSV file and map audio files to unique original song IDs to prepare for song-level splitting.


In [3]:
# Locate dataset path
csv_path = None
audio_base = None

possible_data_roots = [
    '/kaggle/input/datasets/andradaolteanu/gtzan-dataset-music-genre-classification/Data',
    os.path.expanduser('~/.cache/kagglehub/datasets/andradaolteanu/gtzan-dataset-music-genre-classification/versions/1/Data'),
    '../Data',
    'Data'
]

for root in possible_data_roots:
    if os.path.exists(root):
        csv_path = os.path.join(root, 'features_3_sec.csv')
        audio_base = os.path.join(root, 'genres_original')
        break

if csv_path is None or not os.path.exists(csv_path):
    for root_dir, _, files in os.walk(os.path.expanduser('~/.cache/kagglehub')):
        if 'features_3_sec.csv' in files:
            csv_path = os.path.join(root_dir, 'features_3_sec.csv')
            audio_base = os.path.join(root_dir, 'genres_original')
            break

print('CSV Path:', csv_path)
print('Audio Base Directory:', audio_base)

df = pd.read_csv(csv_path)

# Extract unique original song ID (e.g., 'blues.00000.0.wav' -> 'blues.00000')
df['song_id'] = df['filename'].str.replace(r'\.\d+\.wav$', '', regex=True)

print(f'Total 3-sec segment entries: {len(df)}')
print(f'Unique original songs: {df["song_id"].nunique()}')
df[['filename', 'song_id', 'label']].head()


CSV Path: ~/.cache/kagglehub/datasets/andradaolteanu/gtzan-dataset-music-genre-classification/versions/1/Data/features_3_sec.csv
Audio Base Directory: ~/.cache/kagglehub/datasets/andradaolteanu/gtzan-dataset-music-genre-classification/versions/1/Data/genres_original
Total 3-sec segment entries: 9990
Unique original songs: 1000


## 4. Audio Segmentation and MFCC Extraction
Extract 40 MFCC features for every 3-second segment of the 1,000 original songs. Unreadable or corrupted files (such as `jazz.00054.wav`) are safely skipped.


In [4]:
X_mfcc = []
y_mfcc = []
song_ids_mfcc = []
skipped_files = []

for i, (song_id, group) in enumerate(df.groupby('song_id')):
    label = group['label'].iloc[0]
    audio_file = os.path.join(audio_base, label, song_id + '.wav')
    
    try:
        for _, row in group.iterrows():
            segment_no = int(re.search(r'\.(\d+)\.wav$', row['filename']).group(1))
            
            # Load only the 3-second segment slice
            audio, sr = librosa.load(
                audio_file,
                sr=SAMPLE_RATE,
                offset=segment_no * DURATION_PER_SEGMENT,
                duration=DURATION_PER_SEGMENT
            )
            
            if len(audio) < DURATION_PER_SEGMENT * SAMPLE_RATE * 0.9:
                continue
                
            mfcc = librosa.feature.mfcc(
                y=audio,
                sr=SAMPLE_RATE,
                n_mfcc=N_MFCC,
                n_fft=N_FFT,
                hop_length=HOP_LENGTH
            )
            
            # mfcc shape is (40, 130) -> transpose to (130, 40) for sequence modeling
            X_mfcc.append(mfcc.T)
            y_mfcc.append(label)
            song_ids_mfcc.append(song_id)
            
    except Exception as e:
        skipped_files.append(song_id)
        print(f'Skipped corrupted file: {song_id}')
        
    if (i + 1) % 200 == 0:
        print(f'Processed {i + 1}/1000 songs | Extracted segments: {len(X_mfcc)}')

X_mfcc = np.array(X_mfcc, dtype=np.float32)
y_mfcc = np.array(y_mfcc)
song_ids_mfcc = np.array(song_ids_mfcc)

print('\n========== EXTRACTION SUMMARY ==========')
print(f'MFCC Sequence Dataset Shape: {X_mfcc.shape}')
print(f'Labels Shape: {y_mfcc.shape}')
print(f'Unique Songs Processed: {len(np.unique(song_ids_mfcc))}')
print(f'Corrupted Songs Skipped: {len(skipped_files)} {skipped_files}')


Processed 200/1000 songs | Extracted segments: 1998
Processed 400/1000 songs | Extracted segments: 3994
Skipped corrupted file: jazz.00054
Processed 600/1000 songs | Extracted segments: 5982
Processed 800/1000 songs | Extracted segments: 7982
Processed 1000/1000 songs | Extracted segments: 9980

========== EXTRACTION SUMMARY ==========
MFCC Sequence Dataset Shape: (9980, 130, 40)
Labels Shape: (9980,)
Unique Songs Processed: 999
Corrupted Songs Skipped: 1 ['jazz.00054']


## 5. Train / Validation / Test Split
To prevent data leakage, original songs are assigned into 70% train, 15% validation, and 15% test splits. Segments from the same song are confined strictly to one subset.


In [5]:
songs = df[['song_id', 'label']].drop_duplicates()

# 70% Train, 30% Temporary
train_songs, temp_songs = train_test_split(
    songs,
    test_size=0.30,
    random_state=RANDOM_SEED,
    stratify=songs['label']
)

# Split remaining 30% into 15% Validation and 15% Test
val_songs, test_songs = train_test_split(
    temp_songs,
    test_size=0.50,
    random_state=RANDOM_SEED,
    stratify=temp_songs['label']
)

train_ids = set(train_songs['song_id'])
val_ids = set(val_songs['song_id'])
test_ids = set(test_songs['song_id'])

# Segment masks based on song IDs
train_mask = np.array([s in train_ids for s in song_ids_mfcc])
val_mask = np.array([s in val_ids for s in song_ids_mfcc])
test_mask = np.array([s in test_ids for s in song_ids_mfcc])

X_train_raw = X_mfcc[train_mask]
X_val_raw = X_mfcc[val_mask]
X_test_raw = X_mfcc[test_mask]

y_train_raw = y_mfcc[train_mask]
y_val_raw = y_mfcc[val_mask]
y_test_raw = y_mfcc[test_mask]

print(f'Training Songs: {len(train_songs)} | Training Segments: {len(X_train_raw)}')
print(f'Validation Songs: {len(val_songs)} | Validation Segments: {len(X_val_raw)}')
print(f'Testing Songs: {len(test_songs)} | Testing Segments: {len(X_test_raw)}')


Training Songs: 700 | Training Segments: 6980
Validation Songs: 150 | Validation Segments: 1500
Testing Songs: 150 | Testing Segments: 1500


## 6. Feature Scaling
Standardize the 40 MFCC features using `StandardScaler`. The scaler is fitted strictly on the training set and then applied to transform validation and test sets while preserving the sequence shape `(130, 40)`.


In [6]:
n_train, time_steps, n_features = X_train_raw.shape

scaler = StandardScaler()

# Reshape to 2D for feature-wise scaling across the 40 MFCCs
X_train_2d = scaler.fit_transform(X_train_raw.reshape(-1, n_features))
X_val_2d = scaler.transform(X_val_raw.reshape(-1, n_features))
X_test_2d = scaler.transform(X_test_raw.reshape(-1, n_features))

# Reshape back to 3D temporal sequence (samples, 130, 40)
X_train = X_train_2d.reshape(X_train_raw.shape)
X_val = X_val_2d.reshape(X_val_raw.shape)
X_test = X_test_2d.reshape(X_test_raw.shape)

print('X_train shape:', X_train.shape)
print('X_val shape:  ', X_val.shape)
print('X_test shape: ', X_test.shape)


X_train shape: (6980, 130, 40)
X_val shape:   (1500, 130, 40)
X_test shape:  (1500, 130, 40)


## 7. Label Encoding
Encode string genre category labels into numerical class indices.


In [7]:
label_encoder = LabelEncoder()

y_train = label_encoder.fit_transform(y_train_raw)
y_val = label_encoder.transform(y_val_raw)
y_test = label_encoder.transform(y_test_raw)

print('Classes:', label_encoder.classes_)
print('Encoded label distribution (Test set):\n', pd.Series(y_test).value_counts().sort_index().to_dict())


Classes: ['blues' 'classical' 'country' 'disco' 'hiphop' 'jazz' 'metal' 'pop'
 'reggae' 'rock']
Encoded label distribution (Test set):
 {0: 150, 1: 150, 2: 150, 3: 150, 4: 150, 5: 150, 6: 150, 7: 150, 8: 150, 9: 150}


## 8. LSTM Model
Construct a 2-layer Long Short-Term Memory (LSTM) network with Dropout and Dense layers.


In [8]:
model = Sequential([
    Input(shape=(130, 40)),
    LSTM(128, return_sequences=True),
    Dropout(0.3),
    LSTM(64),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(10, activation='softmax')
])

model.summary()


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm (LSTM)                 (None, 130, 128)          86528     
                                                                 
 dropout (Dropout)           (None, 130, 128)          0         
                                                                 
 lstm_1 (LSTM)               (None, 64)                49408     
                                                                 
 dropout_1 (Dropout)         (None, 64)                0         
                                                                 
 dense (Dense)               (None, 64)                4160      
                                                                 
 dropout_2 (Dropout)         (None, 64)                0         
                                                                 
 dense_1 (Dense)             (None, 10)                6

## 9. Training
Compile with the Adam optimizer and Sparse Categorical Crossentropy loss. Train for up to 30 epochs with early stopping based on validation loss.


In [9]:
model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=EARLY_STOPPING_PATIENCE,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop],
    verbose=1
)


Epoch 1/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.3910 - loss: 1.7387 - val_accuracy: 0.4553 - val_loss: 1.4725
Epoch 2/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.5397 - loss: 1.3525 - val_accuracy: 0.5113 - val_loss: 1.3579
Epoch 3/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.5701 - loss: 1.2740 - val_accuracy: 0.5367 - val_loss: 1.3088
Epoch 4/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.6265 - loss: 1.1207 - val_accuracy: 0.5687 - val_loss: 1.2741
Epoch 5/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.6745 - loss: 0.9957 - val_accuracy: 0.5607 - val_loss: 1.2733
Epoch 6/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.6915 - loss: 0.9410 - val_accuracy: 0.5840 - val_loss: 1.2624
Epoch 7/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.7092 - loss: 0.9029 - val_accuracy: 0.5513 - val_loss: 1.3932
Epoch 8/30
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.7497 - loss: 0.7950 - val_accuracy: 0.

## 10. Evaluation
Evaluate the restored optimal LSTM model on the unseen test set across Accuracy, Weighted Precision, Weighted Recall, and Weighted F1-score.


In [10]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=1)

y_pred_prob = model.predict(X_test)
y_pred = y_pred_prob.argmax(axis=1)

precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

print('\n================ TEST EVALUATION RESULTS ================')
print(f'Accuracy : {test_acc:.4f} ({test_acc*100:.2f}%)')
print(f'Precision: {precision:.4f} ({precision*100:.2f}%)')
print(f'Recall   : {recall:.4f} ({recall*100:.2f}%)')
print(f'F1-Score : {f1:.4f} ({f1*100:.2f}%)')
print('==========================================================')

results_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-score'],
    'Score': [f'{test_acc*100:.2f}%', f'{precision*100:.2f}%', f'{recall*100:.2f}%', f'{f1*100:.2f}%']
})
display(results_df)


47/47 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5920 - loss: 1.2580
47/47 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step

================ TEST EVALUATION RESULTS ================
Accuracy : 0.5920 (59.20%)
Precision: 0.5861 (58.61%)
Recall   : 0.5920 (59.20%)
F1-Score : 0.5870 (58.70%)


## 11. Confusion Matrix
Plot and save the multiclass confusion matrix across all 10 genre classes.


In [11]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_
)
plt.title('LSTM Confusion Matrix (Test Set)')
plt.xlabel('Predicted Genre')
plt.ylabel('Actual Genre')
plt.tight_layout()

cm_save_path = os.path.join(RESULTS_DIR, 'lstm_confusion_matrix.png')
plt.savefig(cm_save_path, dpi=300)
plt.show()
print('Confusion matrix saved to:', cm_save_path)


Confusion matrix saved to: results/lstm_confusion_matrix.png


## 12. Learning Curves
Plot and save the Training vs Validation Accuracy and Loss curves.


In [12]:
# Accuracy Plot
plt.figure(figsize=(8, 5))
plt.plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)
plt.plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
plt.title('LSTM Training vs Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()

acc_save_path = os.path.join(RESULTS_DIR, 'lstm_accuracy.png')
plt.savefig(acc_save_path, dpi=300)
plt.show()
print('Accuracy curve saved to:', acc_save_path)

# Loss Plot
plt.figure(figsize=(8, 5))
plt.plot(history.history['loss'], label='Training Loss', linewidth=2)
plt.plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
plt.title('LSTM Training vs Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()

loss_save_path = os.path.join(RESULTS_DIR, 'lstm_loss.png')
plt.savefig(loss_save_path, dpi=300)
plt.show()
print('Loss curve saved to:', loss_save_path)


Accuracy curve saved to: results/lstm_accuracy.png
Loss curve saved to: results/lstm_loss.png


## 13. Error Analysis
Based on the actual test confusion matrix:

### 1. High-Performing Genres
- **Classical**: Classified with the highest accuracy (**143 out of 150 segments**, 95.3%). Its acoustic dynamics and orchestral timbre are distinct from contemporary genres.
- **Metal**: Classified strongly (**118 out of 150 segments**, 78.7%), owing to aggressive distortion and distinct spectral envelope patterns.
- **Pop** (103/150, 68.7%) and **Jazz** (100/150, 66.7%) also demonstrate solid classification reliability.

### 2. Frequently Confused Genres
- **Rock**: The most heavily confused genre (**42 out of 150 segments correct**, 28.0%). It is frequently misclassified as **Country** (29 segments) and **Disco** (24 segments) due to overlapping instrumentation (electric/acoustic guitars, basslines, standard 4/4 drum patterns).
- **Disco**: Frequently confused with **Pop** (29 segments) and **Rock** (25 segments) due to shared rhythmic tempo and synthesizer/guitar arrangements.
- **Country**: Often misclassified as **Rock** (23 segments) and **Classical** (12 segments).
- **Hiphop & Reggae**: Exhibit mutual confusion (19 Hiphop segments predicted as Reggae, 15 Reggae segments predicted as Hiphop) due to similar bass frequencies, tempo ranges, and rhythmic syncopation.
